# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Михальчук Антон'
ГРУПА        = 'ШІ-32'
НОМЕР        = 13

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 13   ·   Михальчук Антон   ·   ШІ-32

Ваші вісім стовпців:
   1. номер і/х
   2. Діагноз
   3. ЛН
   4. Дата розтину
   5. КТ
   6. Прокальцитонін
   7. БРзаг2
   8. МНВ1

Ваша пара для аналізу зв’язку: Перебіг та ступінь КТ

Ваше додаткове питання:
   Стовпець «помер/жив» містить числа 1 і 2. Які три різні статистики pandas порахує для нього без жодної скарги — і чому кожна з них безглузда?


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,Діагноз,ЛН,Дата розтину,КТ,Прокальцитонін,БРзаг2,МНВ1
0,2377/239,Негос дв полісегм пнев 4 кл гр,ІІІ-IV,2022-03-11,20-30%,0.126,17.0,3.10
1,1121/19,Негосп полісег пневм 4 кл гр,ІІ-ІІІ,NaT,30-35%,NaN,13.6,1.22
2,283/674,Негос полісегм пнев 4 кл гр,ІІ,NaT,70-80%,NaN,18.7,1.03
3,2372/513,Негос дв полісегм пнев 4 кл гр,ІІІ-IV,NaT,0.8,NaN,13.6,1.69
4,1441/671,Негос дв полісегм пнев 4 кл гр,ІІ-ІІІ,NaT,NaN,NaN,15.3,1.15
5,2573/354,Негос дв полісегм пнев 4 кл гр,ІІ-ІІІ,2021-12-26,30-40%,NaN,15.3,0.90
6,1762/555,Негос дв полісегм пнев 4 кл гр,ІІ-ІІІ,NaT,60-65%,NaN,15.3,1.09
7,2664/348,Негос дв полісегм пнев 3 кл гр,І,NaT,до 17%,NaN,15.3,0.94


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   номер і/х       121 non-null    object        
 1   Діагноз         121 non-null    str           
 2   ЛН              121 non-null    str           
 3   Дата розтину    61 non-null     datetime64[us]
 4   КТ              44 non-null     object        
 5   Прокальцитонін  46 non-null     float64       
 6   БРзаг2          101 non-null    float64       
 7   МНВ1            90 non-null     float64       
dtypes: datetime64[us](1), float64(3), object(2), str(2)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'"]

--- Діагноз
    непорожніх 121 з 121 | унікальних 15 | типи: str×121
    приклади: ["'Негос дв полісегм пнев 4 кл гр '", "'Негосп полісег пневм 4 кл гр'", "'Негос полісегм пнев 4 кл гр'", "'Негос дв полісегм пнев 3 кл гр '"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'70-80%'", '0.8']

--- Прокальцитонін
    непорожніх 46 з 121 | унікальних 17 | типи: float×46
    приклади: ['np.float64(0.126)', 'np.float64(0.9)', 'np.flo

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'номер і/х':      ('ідентифікатор',
                       'Номер історії хвороби лише позначає запис пацієнта; це не величина, '
                       'тому ні порядок, ні різниці між номерами нічого не означають.'),
    'Діагноз':        ('номінальна',
                       'Назви діагнозів можна лише порівнювати на рівність; природного порядку '
                       'між ними немає (той самий діагноз записано 15 різними написаннями).'),
    'ЛН':             ('порядкова',
                       'Ступінь легеневої недостатності І < ІІ < ІІІ < IV має природний порядок, '
                       'але «відстань» між сусідніми ступенями невідома й неоднакова.'),
    'Дата розтину':   ('інтервальна',
                       'Різниця між датами має сенс (кількість днів), але нуль календаря умовний: '
                       'сказати, що одна дата «удвічі більша» за іншу, не можна.'),
    'КТ':             ('абсолютна',
                       'Частка ураження легень за КТ: є природний нуль (немає ураження) і природна '
                       'одиниця (уся легеня), тому 0.6 справді вдвічі більше за 0.3.'),
    'Прокальцитонін': ('відношень',
                       'Концентрація в нг/мл має природний нуль (речовини немає), тому відношення '
                       'мають сенс: 1.0 нг/мл — у 10 разів більше, ніж 0.1 нг/мл.'),
    'БРзаг2':         ('відношень',
                       'Загальний білірубін (повторний аналіз), мкмоль/л: природний нуль і '
                       'довільна одиниця вимірювання, тож «удвічі вищий» має сенс.'),
    'МНВ1':           ('відношень',
                       'Міжнародне нормалізоване відношення згортання крові — неперервна величина '
                       'з природним нулем, тому відношення та відсотки змін мають сенс.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df


,Стовпець,Шкала,Обґрунтування
0,номер і/х,ідентифікатор,Номер історії хвороби лише позначає запис пацієнта; це н...
1,Діагноз,номінальна,Назви діагнозів можна лише порівнювати на рівність; прир...
2,ЛН,порядкова,Ступінь легеневої недостатності І < ІІ < ІІІ < IV має пр...
3,Дата розтину,інтервальна,"Різниця між датами має сенс (кількість днів), але нуль к..."
4,КТ,абсолютна,Частка ураження легень за КТ: є природний нуль (немає ур...
5,Прокальцитонін,відношень,Концентрація в нг/мл має природний нуль (речовини немає)...
6,БРзаг2,відношень,"Загальний білірубін (повторний аналіз), мкмоль/л: природ..."
7,МНВ1,відношень,Міжнародне нормалізоване відношення згортання крові — не...


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
ПОРЯДОК_ЛН = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']   # порядок задаємо вручну, не сортуванням


def частка_КТ(v):
    """КТ записано змішано: 0.3, '20-30%', 'до 17%'. Повертає частку 0..1 (середина діапазону)."""
    if isinstance(v, (int, float)):
        return v / 100 if v > 1 else float(v)
    s = str(v).replace('%', '').replace('до', '').replace(' ', '')
    if not s:
        return np.nan
    межі = [float(x) for x in s.split('-')]
    return sum(межі) / len(межі) / 100


def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    if стовпець == 'КТ':                       # спершу приводимо змішані записи до частки
        з = з.map(частка_КТ).dropna()
    if pd.api.types.is_datetime64_any_dtype(з):
        з = з.dt.date
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р
    # номінальна і вище: мода й частоти
    р['мода'] = з.mode().iloc[0]
    р['частоти (топ-3)'] = {k: int(v) for k, v in з.value_counts().head(3).items()}
    if ш == 'номінальна':
        return р
    # порядкова: медіана й квартилі за кодами рівнів у заданому вручну порядку
    if ш == 'порядкова':
        коди = pd.Categorical(з, categories=ПОРЯДОК_ЛН, ordered=True).codes
        q1, med, q3 = np.quantile(коди, [0.25, 0.5, 0.75], method='nearest')
        р['медіана'] = ПОРЯДОК_ЛН[med]
        р['квартилі'] = (ПОРЯДОК_ЛН[q1], ПОРЯДОК_ЛН[q3])
        return р
    # інтервальна: ще середнє й стандартне відхилення (для дат — у днях); КВ недопустимий
    if ш == 'інтервальна':
        дати = pd.to_datetime(pd.Series(з))
        р['медіана'] = дати.median().date()
        р['середнє'] = дати.mean().date()
        р['ст. відхилення, днів'] = round(дати.std() / pd.Timedelta(days=1), 1)
        р['розмах, днів'] = (дати.max() - дати.min()).days
        return р
    # відношень і абсолютна: усе, зокрема коефіцієнт варіації та геометричне середнє
    з = з.astype(float)
    р['мода'] = float(р['мода'])
    р.update({'медіана': round(float(з.median()), 3), 'середнє': round(float(з.mean()), 3),
              'ст. відхилення': round(float(з.std()), 3),
              'коеф. варіації, %': round(float(з.std() / з.mean() * 100), 1),
              'геом. середнє': round(float(np.exp(np.log(з[з > 0]).mean())), 3)})
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()


--- номер і/х  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 119}

--- Діагноз  (номінальна)
    {'непорожніх': 121, 'унікальних': 15, 'мода': 'Негосп дв полісегм пнев 4 кл група', 'частоти (топ-3)': {'Негосп дв полісегм пнев 4 кл група': 43, 'НГ дв. Полісег пневмонія 4кл гр.': 19, 'Негосп дв полісегм пнев 3 кл гр': 14}}

--- ЛН  (порядкова)
    {'непорожніх': 121, 'унікальних': 6, 'мода': 'ІІ-ІІІ', 'частоти (топ-3)': {'ІІ-ІІІ': 59, 'ІІ': 21, 'ІІІ': 17}, 'медіана': 'ІІ-ІІІ', 'квартилі': ('ІІ', 'ІІ-ІІІ')}

--- Дата розтину  (інтервальна)
    {'непорожніх': 61, 'унікальних': 50, 'мода': datetime.date(2022, 6, 7), 'частоти (топ-3)': {datetime.date(2022, 6, 7): 3, datetime.date(2022, 4, 7): 2, datetime.date(2022, 4, 27): 2}, 'медіана': datetime.date(2022, 4, 23), 'середнє': datetime.date(2022, 4, 6), 'ст. відхилення, днів': 72.6, 'розмах, днів': 532}

--- КТ  (абсолютна)
    {'непорожніх': 43, 'унікальних': 21, 'мода': 0.3, 'частоти (топ-3)': {0.3: 7, 0.25: 6, 0.8: 3}, 'медіана': 0

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
номер і/х,121.0,119.0,1837.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Діагноз,121,15,Негосп дв полісегм пнев 4 кл група,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Прокальцитонін,46.0,NaN,NaN,NaN,0.752957,0.043,0.1,0.1,0.1205,18.22,3.010375
БРзаг2,101.0,NaN,NaN,NaN,16.216832,7.4,13.6,15.3,18.7,60.5,5.537094
МНВ1,90.0,NaN,NaN,NaN,1.333778,0.8,1.0525,1.19,1.34,10.6,1.032131


In [10]:
номери = pd.to_numeric(df['номер і/х'], errors='coerce')        # pandas візьме ті номери, що схожі на числа
лн_коди = pd.Categorical(df['ЛН'], categories=ПОРЯДОК_ЛН, ordered=True).codes + 1   # І=1 … ІІІ-IV=6
дні = (df['Дата розтину'].dropna() - pd.Timestamp('1970-01-01')).dt.days            # дата як «днів від 1970»

БЕЗГЛУЗДІ = [
    ('номер і/х', 'середнє (65 номерів, схожих на числа)', round(float(номери.mean()), 1),
     'Номер історії хвороби — лише мітка запису, а не величина: «середній номер» нічого не каже '
     'про пацієнтів і зміниться, якщо реєстратура почне нумерацію з іншого числа.'),
    ('ЛН', 'середнє кодів ступенів', round(float(лн_коди.mean()), 2),
     'Шкала порядкова: відомо, що ІІ важче за І, але не наскільки. Кодування 1…6 мовчки припускає '
     'однакові відстані між ступенями, тож середнє 3.57 («між ІІ-ІІІ та ІІІ»?) — артефакт кодування, а не «середній ступінь».'),
    ('Дата розтину', 'коефіцієнт варіації, %', round(float(дні.std() / дні.mean() * 100), 2),
     'Шкала інтервальна: нуль календаря умовний (1970 у pandas, 1900 в Excel), тому КВ = std/mean '
     'залежить від вибору нуля і для тих самих дат може бути і 0.38 % (від 1970), і 15.8 % (від 01.01.2021).'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])


,Стовпець,Статистика,Значення,Чому безглузде
0,номер і/х,"середнє (65 номерів, схожих на числа)",1770.10,"Номер історії хвороби — лише мітка запису, а не величина..."
1,ЛН,середнє кодів ступенів,3.57,"Шкала порядкова: відомо, що ІІ важче за І, але не наскіл..."
2,Дата розтину,"коефіцієнт варіації, %",0.38,Шкала інтервальна: нуль календаря умовний (1970 у pandas...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Перебіг × ступінь КТ

Перебіг: шкала — не у вашому списку
ступінь КТ: шкала — не у вашому списку

  Перебіг ступінь КТ
0  важкий    сердній
1  важкий   середній
2  важкий  критичний
3  важкий  критичний
4  важкий        NaN
5  важкий   середній


In [12]:
ОБРАНА_МІРА = 'рангова кореляція Спірмена'
ЧОМУ_САМЕ_ВОНА = ('Обидві змінні порядкові: «Перебіг» (серед важ < важкий < вкрай важкий) і «ступінь КТ» '
                  '(легкий < середній < важкий < вкрай важкий < критичний) мають порядок, але відстані між рівнями невідомі. '
                  'Спірмен використовує лише ранги, тобто тільки порядок.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

from scipy.stats import spearmanr, pearsonr

# уніфікуємо написання (у даних: «сердній», «серед ступ», «легка», пробіли в кінці)
ЧИСТКА_КТ = {'легка': 'легкий', 'сердній': 'середній', 'серед ступ': 'середній',
             'серед ступеня': 'середній'}
ПОРЯДОК_ПЕРЕБІГ = ['серед важ', 'важкий', 'вкрай важкий']
ПОРЯДОК_КТ = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']   # «вкрай важкий» — окремий рівень

перебіг = df[A].str.strip()
ступінь = df[B].str.strip().replace(ЧИСТКА_КТ)
пара = pd.DataFrame({
    A: pd.Categorical(перебіг, categories=ПОРЯДОК_ПЕРЕБІГ, ordered=True).codes,
    B: pd.Categorical(ступінь, categories=ПОРЯДОК_КТ, ordered=True).codes,
}).replace(-1, np.nan).dropna()

print(pd.crosstab(перебіг, ступінь).reindex(index=ПОРЯДОК_ПЕРЕБІГ, columns=ПОРЯДОК_КТ, fill_value=0))
rho, p = spearmanr(пара[A], пара[B])
print(f'\nСпірмен: rho = {rho:.3f}, p = {p:.3f}, пар спостережень: {len(пара)}')


ступінь КТ    легкий  середній  важкий  вкрай важкий  критичний
Перебіг                                                        
серед важ          6        10       1             0          0
важкий             1         8       9             3          3
вкрай важкий       0         0       0             1          0

Спірмен: rho = 0.630, p = 0.000, пар спостережень: 42


In [13]:
# Контрприклад: Пірсон на кодах рівнів. Він вважає коди справжніми числами з рівними відстанями.
# Закодуємо ті самі рівні двома способами — порядок однаковий, змінюються лише «відстані».
код_1 = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5}     # легкий=1 … критичний=5
код_2 = {0: 1, 1: 2, 2: 3, 3: 4, 4: 10}    # той самий порядок, але «критичний» далеко
for назва, код in [('коди 1-2-3-4-5 ', код_1), ('коди 1-2-3-4-10', код_2)]:
    r_p, _ = pearsonr(пара[A], пара[B].map(код))
    r_s, _ = spearmanr(пара[A], пара[B].map(код))
    print(f'{назва}: Пірсон = {r_p:.3f} | Спірмен = {r_s:.3f}')

print('\nПорядок рівнів той самий, але Пірсон змінився — він залежить від довільно обраних кодів.')
print('Спірмен однаковий: йому важливий лише порядок. Тому число Пірсона тут не можна інтерпретувати.')


коди 1-2-3-4-5 : Пірсон = 0.596 | Спірмен = 0.630
коди 1-2-3-4-10: Пірсон = 0.421 | Спірмен = 0.630

Порядок рівнів той самий, але Пірсон змінився — він залежить від довільно обраних кодів.
Спірмен однаковий: йому важливий лише порядок. Тому число Пірсона тут не можна інтерпретувати.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'номер і/х':      'Не беремо в модель: ідентифікатор не несе інформації, а модель може знайти в номерах випадкову закономірність.',
    'Діагноз':        'Зводимо 15 написань до двох значень (3 чи 4 клінічна група), потім one-hot; порядок категорій не вводимо.',
    'ЛН':             'Цілі коди в порядку, заданому вручну (І=0 … ІІІ-IV=5): автоматичне сортування рядків не розуміє зміст римських цифр.',
    'Дата розтину':   'Не беремо в модель: дата є лише в померлих, тож це витік цільової змінної («спойлер» результату).',
    'КТ':             'Лише переводимо змішані записи («20-30%», 0.3) у частку 0…1; додатково масштабувати не треба.',
    'Прокальцитонін': 'Логарифм: розподіл дуже несиметричний (більшість ≈0.1, викиди 10 і 18.22), логарифм стискає викиди.',
    'БРзаг2':         'Стандартизація (z = (x − середнє) / ст. відхилення): розподіл майже симетричний, середнє ≈ медіані.',
    'МНВ1':           'Логарифм: розподіл несиметричний (медіана 1.19, максимум 10.6), як у прокальцитоніну.',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])


,Стовпець,Шкала,Як готуємо
0,номер і/х,ідентифікатор,"Не беремо в модель: ідентифікатор не несе інформації, а ..."
1,Діагноз,номінальна,Зводимо 15 написань до двох значень (3 чи 4 клінічна гру...
2,ЛН,порядкова,"Цілі коди в порядку, заданому вручну (І=0 … ІІІ-IV=5): а..."
3,Дата розтину,інтервальна,"Не беремо в модель: дата є лише в померлих, тож це витік..."
4,КТ,абсолютна,"Лише переводимо змішані записи («20-30%», 0.3) у частку ..."
5,Прокальцитонін,відношень,"Логарифм: розподіл дуже несиметричний (більшість ≈0.1, в..."
6,БРзаг2,відношень,Стандартизація (z = (x − середнє) / ст. відхилення): роз...
7,МНВ1,відношень,"Логарифм: розподіл несиметричний (медіана 1.19, максимум..."


In [15]:
def стандартизувати(з):
    """z-оцінка; сталий стовпець дав би ділення на нуль і мовчазні NaN — зупиняємося явно."""
    if з.std() == 0 or з.nunique() <= 1:
        raise ValueError(f'«{з.name}» — сталий стовпець, стандартизація неможлива')
    return (з - з.mean()) / з.std()


# 1) номінальна: Діагноз → дві клінічні групи → one-hot
група = df['Діагноз'].str.extract(r'(\d)\s*кл', expand=False).map({'3': '3 кл. гр.', '4': '4 кл. гр.'})
діагноз_onehot = pd.get_dummies(група, prefix='Діагноз', dtype=int)
print('Діагноз:', df['Діагноз'].nunique(), 'написань →', група.nunique(), 'групи →', list(діагноз_onehot.columns))

# 2) порядкова: ЛН → цілі коди в заданому вручну порядку
лн_код = pd.Series(pd.Categorical(df['ЛН'].str.strip(), categories=ПОРЯДОК_ЛН, ordered=True).codes,
                   index=df.index, name='ЛН_код')
assert (лн_код >= 0).all(), 'є значення ЛН поза заданим порядком'
print('ЛН:', dict(zip(ПОРЯДОК_ЛН, range(len(ПОРЯДОК_ЛН)))))

# 3) відношень: Прокальцитонін і МНВ1 → логарифм; БРзаг2 → стандартизація
assert (df['Прокальцитонін'].dropna() > 0).all() and (df['МНВ1'].dropna() > 0).all()
пкт_лог = np.log(df['Прокальцитонін'])
мнв_лог = np.log(df['МНВ1'])
бр_z = стандартизувати(df['БРзаг2'])

підготовлено = pd.concat([діагноз_onehot, лн_код, пкт_лог.rename('ПКТ_log'),
                          мнв_лог.rename('МНВ1_log'), бр_z.rename('БРзаг2_z')], axis=1)
print()
print(підготовлено.describe().round(2).loc[['count', 'mean', 'std', 'min', 'max']].to_string())


Діагноз: 15 написань → 2 групи → ['Діагноз_3 кл. гр.', 'Діагноз_4 кл. гр.']
ЛН: {'І': 0, 'І-ІІ': 1, 'ІІ': 2, 'ІІ-ІІІ': 3, 'ІІІ': 4, 'ІІІ-IV': 5}

       Діагноз_3 кл. гр.  Діагноз_4 кл. гр.  ЛН_код  ПКТ_log  МНВ1_log  БРзаг2_z
count             121.00             121.00  121.00    46.00     90.00    101.00
mean                0.25               0.75    2.57    -1.93      0.21     -0.00
std                 0.43               0.43    1.20     1.11      0.31      1.00
min                 0.00               0.00    0.00    -3.15     -0.22     -1.59
max                 1.00               1.00    5.00     2.90      2.36      8.00


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    (1, 'Середнє для номінальної шкали: «помер/жив» = 1/2 — це ярлики, а не кількість.',
     'Значення 1.50 залежить від довільного кодування (з кодами 0/1 чи 1/5 вийшло б інше); '
     '«пацієнт посередині між смертю й життям» не має змісту.',
     'Частоти й частка: померло 61 з 121 (летальність 50.4 %), вижило 60; мода — «помер».'),
    (2, 'Дані не очищено: «ж»/«жін» і «ч»/«чол» — однакові категорії, записані по-різному.',
     '«4 статі» — артефакт написання, а не властивість вибірки; частки груп (46 % «ж») неправильні.',
     'Уніфікувати написання до «ж»/«ч»: жінок 75 (62 %), чоловіків 46 (38 %).'),
    (3, 'Порядкову шкалу відсортовано за абеткою, а потім усереднено коди; написання не очищено.',
     '«Найлегший — важкий, найважчий — середній» — це абетковий порядок, а не тяжкість; середнє кодів '
     'порядкової шкали не має змісту навіть за правильного порядку.',
     'Уніфікувати написання, задати порядок вручну (легкий < середній < важкий < вкрай важкий < критичний), '
     'описати частотами й медіаною.'),
    (4, 'Обчислено на непридатних даних: errors="coerce" мовчки перетворив текстові записи («70-80%», «до 17%») на NaN.',
     '39.6 % пораховано лише по 27 пацієнтах із 43; викинуто переважно великі значення, тож середнє занижене.',
     'Спершу перевести всі записи в частку (середина діапазону) — середнє 42.5 % по 43 пацієнтах.'),
    (5, 'Змішані формати запису: одне значення 0.9 — це 90 %, записані часткою.',
     'Мінімум 0.9 % означав би несумісний із життям стан; це значення тягне середнє 82.05 вниз.',
     'Привести 0.9 до 90 перед обчисленням (або описувати медіаною, 84, яка до нього стійка).'),
    (6, 'Коефіцієнт варіації для інтервальної шкали (дати): нуль календаря умовний.',
     'КВ = std/mean залежить від вибору нуля (тут 1970 р., ≈20 000 днів), тому 0.595 % нічого не каже '
     'про однорідність потоку госпіталізацій — від іншого нуля вийшли б десятки відсотків.',
     'Описувати розкид дат стандартним відхиленням у днях або розмахом (допустимі для інтервальної шкали).'),
    (7, 'Кореляція Пірсона для двох номінальних змінних (стать × помер/жив).',
     'Пірсон міряє лінійний зв’язок між величинами; у статі й результату лікування немає ні величини, ні напрямку, '
     'тож висновок про «відсутність лінійного зв’язку» не має змісту.',
     'Таблиця спряженості, χ² і коефіцієнт Крамера V.'),
    (8, 'Нормалізовано всі «числові» стовпці без перевірки шкали й варіативності.',
     'ФЕРИТ2 заповнений лише в одного пацієнта (сталий): max = min, ділення 0/0 дає мовчазні NaN; '
     '«помер/жив» — номінальна цільова змінна, її нормалізація безглузда.',
     'Нормалізувати лише стовпці шкали відношень, перевіривши max ≠ min; сталі стовпці викинути, ціль не чіпати.'),
    (9, 'dropna() викидає рядок за будь-який пропуск, а пропуски в цьому файлі структурні.',
     'Дата розтину порожня в усіх живих, ФЕРИТ2 — у 120 пацієнтів, тож лишилося 0 рядків: '
     '«повністю заповнений набір» насправді порожній.',
     'Спершу прибрати стовпці зі змістовними чи масовими пропусками, потім dropna(subset=[потрібні ознаки]) '
     'або заповнити пропуски.'),
    (10, 'Витік цільової змінної: дата розтину існує лише в померлих.',
     'r = 1.00 означає не «найкращий предиктор», а те, що ознака записана після смерті; до того ж NaT '
     'перетворено на -9.2e18, тож усі живі отримали одне штучне число.',
     'Виключити дату розтину (і все, що відоме лише після результату) з ознак; рейтинг будувати мірою, '
     'що відповідає шкалі кожної ознаки.'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])


,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Середнє для номінальної шкали: «помер/жив» = 1/2 — це яр...,Значення 1.50 залежить від довільного кодування (з кодам...,"Частоти й частка: померло 61 з 121 (летальність 50.4 %),..."
1,2,Дані не очищено: «ж»/«жін» і «ч»/«чол» — однакові катего...,"«4 статі» — артефакт написання, а не властивість вибірки...","Уніфікувати написання до «ж»/«ч»: жінок 75 (62 %), чолов..."
2,3,"Порядкову шкалу відсортовано за абеткою, а потім усередн...","«Найлегший — важкий, найважчий — середній» — це абеткови...","Уніфікувати написання, задати порядок вручну (легкий < с..."
3,4,"Обчислено на непридатних даних: errors=""coerce"" мовчки п...",39.6 % пораховано лише по 27 пацієнтах із 43; викинуто п...,Спершу перевести всі записи в частку (середина діапазону...
4,5,"Змішані формати запису: одне значення 0.9 — це 90 %, зап...",Мінімум 0.9 % означав би несумісний із життям стан; це з...,Привести 0.9 до 90 перед обчисленням (або описувати меді...
5,6,Коефіцієнт варіації для інтервальної шкали (дати): нуль ...,"КВ = std/mean залежить від вибору нуля (тут 1970 р., ≈20...",Описувати розкид дат стандартним відхиленням у днях або ...
6,7,Кореляція Пірсона для двох номінальних змінних (стать × ...,Пірсон міряє лінійний зв’язок між величинами; у статі й ...,"Таблиця спряженості, χ² і коефіцієнт Крамера V."
7,8,Нормалізовано всі «числові» стовпці без перевірки шкали ...,ФЕРИТ2 заповнений лише в одного пацієнта (сталий): max =...,"Нормалізувати лише стовпці шкали відношень, перевіривши ..."
8,9,"dropna() викидає рядок за будь-який пропуск, а пропуски ...","Дата розтину порожня в усіх живих, ФЕРИТ2 — у 120 пацієн...",Спершу прибрати стовпці зі змістовними чи масовими пропу...
9,10,Витік цільової змінної: дата розтину існує лише в померлих.,"r = 1.00 означає не «найкращий предиктор», а те, що озна...","Виключити дату розтину (і все, що відоме лише після резу..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
# Розділ 1: середнє номінальної шкали → частоти
print('РОЗДІЛ 1')
print('  хибно   : середнє «помер/жив» =', round(df['помер/жив'].mean(), 2))
частоти = df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts()
print('  правильно:', частоти.to_dict(), f"→ летальність {частоти['помер'] / len(df):.1%}")

# Розділ 4: coerce викидає текстові записи → спершу переводимо все в частку
print('\nРОЗДІЛ 4')
кт_хибно = pd.to_numeric(df['КТ'], errors='coerce')
кт_правильно = df['КТ'].dropna().map(частка_КТ).dropna()
print(f'  хибно   : середнє {100 * кт_хибно.mean():.1f} % по {кт_хибно.count()} пацієнтах')
print(f'  правильно: середнє {100 * кт_правильно.mean():.1f} % по {len(кт_правильно)} пацієнтах')

# Розділ 9: dropna() по всіх стовпцях → лише по потрібних ознаках
print('\nРОЗДІЛ 9')
ознаки = ['Вік', 'ЛН', 'spO2 без дот', 'БРзаг2', 'МНВ1']
print('  хибно   : df.dropna() → рядків:', len(df.dropna()))
print('  правильно: df.dropna(subset=ознаки) → рядків:', len(df.dropna(subset=ознаки)), '| ознаки:', ознаки)


РОЗДІЛ 1
  хибно   : середнє «помер/жив» = 1.5
  правильно: {'помер': 61, 'вижив': 60} → летальність 50.4%

РОЗДІЛ 4
  хибно   : середнє 39.6 % по 27 пацієнтах
  правильно: середнє 42.5 % по 43 пацієнтах

РОЗДІЛ 9
  хибно   : df.dropna() → рядків: 0
  правильно: df.dropna(subset=ознаки) → рядків: 74 | ознаки: ['Вік', 'ЛН', 'spO2 без дот', 'БРзаг2', 'МНВ1']


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    'КТ записано в трьох форматах (0.3, «20-30%», «до 17%»): pd.to_numeric(errors="coerce") мовчки '
    'відкидає 16 із 43 записів, переважно великі діапазони, і занижує середнє ураження легень до 39.6 % '
    'замість 42.5 % після переведення всіх записів у частку.',
    'Дата розтину заповнена рівно в усіх 61 померлого й порожня в усіх 60 живих: пропуск тут не помилка, '
    'а означає «вижив», тому ця ознака — витік цільової змінної і в модель прогнозу летальності її брати не можна.',
    'Діагноз має 15 різних написань, але по суті містить лише дві категорії — пневмонію 4 клінічної групи '
    '(91 пацієнт) і 3 клінічної групи (30): без уніфікації частоти й one-hot кодування дали б 15 фіктивних категорій.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. КТ записано в трьох форматах (0.3, «20-30%», «до 17%»): pd.to_numeric(errors="coerce") мовчки відкидає 16 із 43 записів, переважно великі діапазони, і занижує середнє ураження легень до 39.6 % замість 42.5 % після переведення всіх записів у частку.
2. Дата розтину заповнена рівно в усіх 61 померлого й порожня в усіх 60 живих: пропуск тут не помилка, а означає «вижив», тому ця ознака — витік цільової змінної і в модель прогнозу летальності її брати не можна.
3. Діагноз має 15 різних написань, але по суті містить лише дві категорії — пневмонію 4 клінічної групи (91 пацієнт) і 3 клінічної групи (30): без уніфікації частоти й one-hot кодування дали б 15 фіктивних категорій.


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'Стовпець «помер/жив» номінальний: 1 і 2 — лише ярлики категорій «помер» і «вижив», їх можна було б '
    'закодувати будь-якими іншими числами чи літерами. Pandas бачить числа й без скарги рахує, зокрема:\n'
    '1) середнє (mean = 1.50) — «пацієнт посередині між смертю й життям» не має змісту, а з кодами 0/1 '
    'чи 1/5 «середнє» було б іншим;\n'
    '2) стандартне відхилення (std ≈ 0.50) — «розкид результату лікування» залежить лише від того, якими '
    'числами закодовано категорії: з кодами 0/1 чи 1/10 він був би іншим для тих самих даних;\n'
    '3) мінімум і максимум (min = 1, max = 2) — припускають порядок «менше–більше», якого між «помер» і '
    '«вижив» немає: якщо поміняти коди місцями, «мінімум» стане «максимумом».\n'
    'Коректно описувати такий стовпець частотами й модою: померло 61 з 121 (летальність 50.4 %), вижило 60.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «помер/жив» містить числа 1 і 2. Які три різні статистики pandas порахує для нього без жодної скарги — і чому кожна з них безглузда?

Стовпець «помер/жив» номінальний: 1 і 2 — лише ярлики категорій «помер» і «вижив», їх можна було б закодувати будь-якими іншими числами чи літерами. Pandas бачить числа й без скарги рахує, зокрема:
1) середнє (mean = 1.50) — «пацієнт посередині між смертю й життям» не має змісту, а з кодами 0/1 чи 1/5 «середнє» було б іншим;
2) стандартне відхилення (std ≈ 0.50) — «розкид результату лікування» залежить лише від того, якими числами закодовано категорії: з кодами 0/1 чи 1/10 він був би іншим для тих самих даних;
3) мінімум і максимум (min = 1, max = 2) — припускають порядок «менше–більше», якого між «помер» і «вижив» немає: якщо поміняти коди місцями, «мінімум» стане «максимумом».
Коректно описувати такий стовпець частотами й модою: померло 61 з 121 (летальність 50.4 %), вижило 60.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'код, перевірено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'код, перевірено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Інструмент: Claude (Claude Code). Шкали всіх восьми стовпців, вибір трьох безглуздих чисел, міру зв’язку '
    '(Спірмен, «вкрай важкий» як окремий рівень), способи підготовки кожного стовпця, усі десять помилок звіту '
    'та три висновки я визначив сам у діалозі; ШІ написало код і сформулювало тексти пояснень з моїх відповідей. '
    'Перевірив: частоти «помер/жив» і збіг дати розтину з померлими (61/61), сталість ФЕРИТ2, 0 рядків після '
    'dropna(), середнє КТ 39.6 % проти 42.5 %, зміну Пірсона при іншому кодуванні та незмінність Спірмена.'
)
ПІДТВЕРДЖУЮ = True      # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту


In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 13 готова до захисту.
Михальчук Антон, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.